# تفوّق — صوت الأستاذة بالدارجة الجزائرية (مجاناً على Google Colab)

هذا الدفتر يصنع **حزمة صوت** للمنصة من تسجيلك أنت، على GPU مجاني من Google. تسجيلك يبقى في حسابك على Google Drive، ولا يمرّ بأي مكان آخر.

**قبل البدء:** Runtime ← Change runtime type ← **T4 GPU** ← Save.

**ما تحتاجينه:**
1. `tafawoq-voice-kit.zip` (أرسلته لك).
2. تسجيلك الصوتي (mp3 أو m4a).

ثم شغّلي الخلايا بالترتيب بزر ▶ في كل خلية.

In [ ]:
# 1) Google Drive: هنا يُحفظ العمل، فإذا انقطعت الجلسة نكمل من حيث توقفنا.
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/tafawoq-voice'
import os; os.makedirs(WORK, exist_ok=True); print('Work folder:', WORK)

In [ ]:
# 2) التثبيت (دقيقتان تقريباً)
!nvidia-smi -L || echo 'لا يوجد GPU: اختاري T4 GPU من Runtime ثم أعيدي التشغيل'
!pip -q install habibi-tts==0.1.1 soundfile faster-whisper

In [ ]:
# 3) ارفعي الملفين: tafawoq-voice-kit.zip وتسجيلك الصوتي
from google.colab import files
import shutil, zipfile
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'{WORK}/{name}')
    if name.endswith('.zip'):
        zipfile.ZipFile(f'{WORK}/{name}').extractall(WORK)
print(os.listdir(WORK))

In [ ]:
# 4) اختاري التسجيل، واسمعي ما كتبه البرنامج من أول 10 ثوانٍ
RECORDING = [f'{WORK}/{n}' for n in os.listdir(WORK) if n.lower().endswith(('.mp3', '.m4a', '.wav', '.ogg', '.aac'))][0]
START, SECONDS = 0.4, 10.0
!ffmpeg -hide_banner -loglevel error -y -i "$RECORDING" -ss {START} -t {SECONDS} -ac 1 -ar 24000 /content/ref.wav
from faster_whisper import WhisperModel
segments, _ = WhisperModel('medium', device='cuda', compute_type='float16').transcribe('/content/ref.wav', language='ar')
print('النص المقترح:', ' '.join(s.text.strip() for s in segments))
from IPython.display import Audio, display; display(Audio('/content/ref.wav'))

### 5) صحّحي النص
اكتبي في الخلية التالية **بالضبط** ما تقولينه في الثواني العشر التي سمعتِها (البرنامج يخطئ أحياناً). بالحروف العربية، والأرقام بالكلمات.

In [ ]:
REF_TEXT = 'السلام عليكم، أنا أستاذتكم من منصة تفوق. راح نبنيو مع بعض الدرس، ونفهمو كامل المحاور بتطبيق التمارين.'

In [ ]:
# 6) عيّنة: اسمعي صوتك قبل صنع الحزمة كاملة
import time, torch, soundfile as sf
from huggingface_hub import hf_hub_download
from f5_tts.infer.utils_infer import load_model, load_vocoder, preprocess_ref_audio_text
from f5_tts.model import DiT
from habibi_tts.infer.utils_infer import infer_process
ckpt = hf_hub_download('samifalouti/TassiliTTS', 'ALGERIA.safetensors'); vocab = hf_hub_download('samifalouti/TassiliTTS', 'vocab.txt')
vocoder = load_vocoder(vocoder_name='vocos', is_local=False, device='cuda')
model = load_model(DiT, dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4), ckpt_path=ckpt, vocab_file=vocab, device='cuda')
ra, rt = preprocess_ref_audio_text('/content/ref.wav', REF_TEXT)
for i, text in enumerate(['صحيت، هاديك هي الإجابة الصحيحة', 'إف مشتقة إكس يساوي اثنان إكس زائد ثلاثة', 'ماعليش، خمم معايا شوية ونعاودو مع بعض']):
    t = time.time()
    audio, sr, _ = infer_process(ref_audio=ra, ref_text=rt, gen_text=text, model_obj=model, vocoder=vocoder, nfe_step=32, device='cuda')
    sf.write(f'/content/sample_{i}.wav', audio, sr)
    print(f'{text}  —  {time.time()-t:.1f}s'); display(Audio(f'/content/sample_{i}.wav'))
del model; torch.cuda.empty_cache()

### 7) الحزمة الكاملة
إذا أعجبك الصوت، شغّلي الخلية التالية. تبدأ بمفردات الرياضيات ثم الجمل. إذا انقطعت الجلسة، أعيدي تشغيل الخلايا 1 و2 و4 و5 ثم هذه الخلية: تكمل من حيث توقفت.

In [ ]:
!python {WORK}/render_tassili.py --manifest {WORK}/manifest.jsonl --reference "$RECORDING" --reference-start {START} --reference-seconds {SECONDS} --reference-text "{REF_TEXT}" --out {WORK}/wavs --hours 3.5

### 8) عند الانتهاء
الملف `voice-pack-wavs.zip` موجود في مجلد **tafawoq-voice** على Google Drive. أرسليه لي، وأستورده في المنصة كصوت الأستاذة الجزائري.